# 02 CPU と GPU（PyTorch）

## 1. 研究目的（RQ3）

PyTorch を使って、**同じ Tensor 演算（行列積）**を CPU と GPU で実行し、

- 結果が一致すること
- 実行時間（データ転送・初回・ウォームアップ後の計算・結果の取り出しを区別して）
- **GPU が有利になる条件・不利になる条件**（行列サイズ、バッチサイズ）

を調べます。GPU がないランタイムでは、CPU の部分だけを実行し、GPU の部分は**スキップ**します（GPU の値は表示しません）。

## 2. 理論的背景

| | CPU | GPU（NVIDIA、CUDA） |
| --- | --- | --- |
| コア | 少数（数個〜数十）の高性能なコア | 数千の単純なコア |
| 得意な処理 | 分岐、逐次処理、小さな計算 | 同じ計算を大量のデータに一斉に（行列積、要素ごとの演算） |
| メモリ | メインメモリ（RAM） | GPU 専用のメモリ（VRAM）。CPU から**転送**が必要 |
| 実行のしかた | 命令を順に実行 | **非同期**：CPU は命令を GPU に送ったらすぐ次に進む |

### 行列積の計算量

$n \times n$ の行列積は、乗算・加算が $n^3$ 回、データは $3n^2$ 個です。$n$ が大きいほど「データ 1 個あたりの計算量」が増えるので、計算の速い GPU が有利になりやすくなります。小さい $n$ では、転送や GPU への命令の送り出し（カーネル起動）の固定コストが目立ちます。

### 時間測定の注意（非同期実行）

GPU の計算は非同期なので、`time.perf_counter()` で単純に挟むと「命令を出した時間」しか測れません。
**`torch.cuda.synchronize()` で GPU の計算が終わるのを待ってから**時計を止めます（`benchmark_utils.measure` の `sync` 引数）。

> 参考 Notebook（`A_cpu_gpu_tpu_benchmark/starter.ipynb`）の GPU 行列積のセルも、`torch.cuda.synchronize()` で前後を同期していました。この Notebook ではそれに加えて、ウォームアップ・複数回測定・転送時間の分離を行います。

## 3. 実験環境（最初に必ず実行）

このセルは、E2 リポジトリの `.py`（`cpu_basics.py`、`numpy_benchmark.py` など）を import できるように、
**作業ディレクトリ（E2 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E2 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E2 フォルダを ZIP にして `/content` にアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/`（教科書のコード）は**公開リポジトリには含めない**約束です。
> `textbook/` がないと `algorithm_benchmark.py` の import が失敗します（Notebook 04・05 の一部）。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e2-computing-performance"   # リポジトリの中の E2 フォルダ（ルートが E2 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e2_root(p):
    """cpu_basics.py と benchmark_utils.py があるフォルダを E2 のルートとみなす"""
    return (Path(p) / "cpu_basics.py").exists() and (Path(p) / "benchmark_utils.py").exists()


def find_e2_root():
    # 1) ローカル：notebooks/ の中、または E2 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e2_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e2_upload")
    for hit in glob.glob("/content/e2_upload/**/cpu_basics.py", recursive=True):
        if is_e2_root(Path(hit).parent):
            return Path(hit).parent
    return None


E2_ROOT = find_e2_root()
if E2_ROOT is None:
    raise RuntimeError("E2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E2_ROOT)
if str(E2_ROOT) not in sys.path:
    sys.path.insert(0, str(E2_ROOT))
print("E2_ROOT:", E2_ROOT)
if not (E2_ROOT / "textbook" / "bsearch.py").exists():
    print("⚠️ textbook/ がありません。algorithm_benchmark.py の import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.6), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は「実装（ライブラリ）」ごとに固定する。デバイス（CPU / GPU / TPU）は線の種類と記号で区別する
COLORS = {"Python": "#eb6834", "NumPy": "#2a78d6", "PyTorch": "#1baf7a", "JAX": "#6250d6",
          "TensorFlow": "#eda100", "neutral": "#8a8985"}
DEVICE_STYLE = {"CPU": ("-", "o"), "GPU": ("--", "s"), "TPU": ("-.", "^")}

In [ ]:
import platform

def show_environment():
    """Python・ライブラリのバージョン、CPU、GPU・TPU の有無を表示する（測定はしない）"""
    info = {"Python": platform.python_version(), "Platform": platform.platform()}
    cpu = platform.processor() or platform.machine()
    try:
        with open("/proc/cpuinfo") as f:
            cpu = next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except (OSError, StopIteration):
        pass
    info["CPU"] = f"{cpu} ({os.cpu_count()} logical cores)"
    try:
        import numpy
        info["NumPy"] = numpy.__version__
    except ImportError:
        info["NumPy"] = "not installed"
    try:
        import torch
        info["PyTorch"] = torch.__version__
        info["CUDA GPU"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available"
    except ImportError:
        info["PyTorch"] = "not installed"
        info["CUDA GPU"] = "unknown (PyTorch not installed)"
    try:
        import jax
        info["JAX"] = jax.__version__
        info["JAX devices"] = ", ".join(f"{d.platform}:{d.device_kind}" for d in jax.devices())
        info["TPU"] = "available" if any(d.platform == "tpu" for d in jax.devices()) else "not available"
    except ImportError:
        info["JAX"] = "not installed"
        info["TPU"] = "unknown (JAX not installed)"
    except Exception as e:   # TPU の初期化に失敗した場合など
        info["JAX"] = info.get("JAX", "installed") + f" (device error: {e})"
        info["TPU"] = "not available"
    for k, v in info.items():
        print(f"{k:12}: {v}")
    return info

ENV = show_environment()

## 4. 使用するコード

- `benchmark_utils.measure`：ウォームアップ＋複数回測定（`sync` で GPU の終了を待つ）
- PyTorch：`torch.cuda.is_available()`、`tensor.to(device)`、`@`（`torch.matmul`）、`torch.bmm`（バッチ行列積）

PyTorch は Colab に標準で入っています。ローカルでなければ、この Notebook の測定部分はスキップされます。

In [ ]:
import time
import numpy as np
from benchmark_utils import measure, format_table

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("PyTorch がインストールされていないので、この Notebook の PyTorch の部分はスキップします。")
    print("（ローカルの CPU で試す例：pip install torch --index-url https://download.pytorch.org/whl/cpu）")

HAS_GPU = HAS_TORCH and torch.cuda.is_available()
DEVICES = (["cpu"] + (["cuda"] if HAS_GPU else [])) if HAS_TORCH else []
LABEL = {"cpu": "CPU", "cuda": "GPU"}
print("使うデバイス:", [LABEL[d] for d in DEVICES] or "なし")
if HAS_TORCH and not HAS_GPU:
    print("GPU（CUDA）は使えません。GPU の測定はスキップします（値は表示しません）。")
if HAS_GPU:
    torch.backends.cuda.matmul.allow_tf32 = False   # TF32（精度を下げて速くする機能）を使わず float32 で比べる
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 0
DTYPE = np.float32
WARMUP, REPEATS = 3, 10
MAX_MATRIX_N = 4096        # 行列サイズの安全上限（4096×4096 の float32 は 1 個 64 MB）

## 5. 実行例

### 5-1. Tensor の生成とデバイス

In [ ]:
if HAS_TORCH:
    t = torch.arange(6, dtype=torch.float32).reshape(2, 3)
    print(t, "device:", t.device, "dtype:", t.dtype)
    if HAS_GPU:
        t_gpu = t.to("cuda")                   # CPU → GPU へ転送
        print("on GPU:", t_gpu.device)
        print("back to CPU:", t_gpu.cpu())    # GPU → CPU へ転送

### 5-2. 1 つの行列サイズで、時間を 4 つに分けて測る

| 記録名 | 意味 |
| --- | --- |
| `data_gen_s` | NumPy で乱数行列を作る時間（CPU） |
| `to_device_s` | NumPy 配列を Tensor にしてデバイスへ送る時間（GPU では CPU → GPU の転送） |
| `first_s` | 最初の 1 回の行列積（初期化などを含み得る） |
| `compute_s` | ウォームアップ後の行列積の中央値（**計算本体**） |
| `to_host_s` | 結果を CPU に取り出す時間（GPU → CPU の転送） |

In [ ]:
def bench_matmul(n, device, seed=SEED):
    """n×n 行列積を device で測り、NumPy の結果と比べる。結果の辞書を返す"""
    if n > MAX_MATRIX_N:
        raise ValueError(f"n = {n} は安全上限 {MAX_MATRIX_N} を超えています")
    sync = (lambda _r=None: torch.cuda.synchronize()) if device == "cuda" else None

    t0 = time.perf_counter()
    rng = np.random.default_rng(seed)
    A = rng.standard_normal((n, n)).astype(DTYPE)
    B = rng.standard_normal((n, n)).astype(DTYPE)
    data_gen = time.perf_counter() - t0

    t0 = time.perf_counter()
    tA, tB = torch.as_tensor(A).to(device), torch.as_tensor(B).to(device)
    if sync: sync()
    to_device = time.perf_counter() - t0

    t0 = time.perf_counter()
    tC = tA @ tB
    if sync: sync()
    first = time.perf_counter() - t0

    compute = measure(lambda: tA @ tB, repeats=REPEATS, warmup=WARMUP, sync=sync)

    t0 = time.perf_counter()
    C = tC.cpu().numpy()
    to_host = time.perf_counter() - t0

    ok = np.allclose(C, A @ B, rtol=1e-3, atol=1e-3 * np.sqrt(n))   # float32 の丸め誤差を考慮
    return {"device": LABEL[device], "n": n, "dtype": np.dtype(DTYPE).name, "repeats": REPEATS,
            "data_gen_s": data_gen, "to_device_s": to_device, "first_s": first,
            "compute_s": compute["median"], "compute_stdev_s": compute["stdev"],
            "to_host_s": to_host, "match_numpy": bool(ok)}


if HAS_TORCH:
    one = [bench_matmul(512, d) for d in DEVICES]
    print(format_table(one, [("device", "device"), ("n", "n"), ("data_gen_s", "data gen"),
                             ("to_device_s", "to device"), ("first_s", "first"),
                             ("compute_s", "compute"), ("to_host_s", "to host"), ("match_numpy", "match")]))
    assert all(r["match_numpy"] for r in one)

### 5-3. 行列サイズを変える

参考 Notebook の `SIZE = 1024` を含む、いくつかのサイズで測ります。CPU の NumPy（`A @ B`）も同じ形式で記録します。

In [ ]:
MATRIX_SIZES = [32, 64, 128, 256, 512, 1024]   # TODO：変えてみよう（安全上限 MAX_MATRIX_N）

def bench_numpy(n, seed=SEED):
    rng = np.random.default_rng(seed)
    A = rng.standard_normal((n, n)).astype(DTYPE)
    B = rng.standard_normal((n, n)).astype(DTYPE)
    r = measure(lambda: A @ B, repeats=REPEATS, warmup=WARMUP)
    return {"device": "CPU", "impl": "NumPy", "n": n, "compute_s": r["median"]}

size_rows = [bench_numpy(n) for n in MATRIX_SIZES]
if HAS_TORCH:
    for n in MATRIX_SIZES:
        for d in DEVICES:
            size_rows.append({**bench_matmul(n, d), "impl": "PyTorch"})
print(format_table(size_rows, [("impl", "impl"), ("device", "device"), ("n", "n"),
                               ("to_device_s", "to device"), ("compute_s", "compute"), ("to_host_s", "to host")]))

### 5-4. バッチサイズを変える（バッチ行列積 `torch.bmm`）

小さな行列（$128 \times 128$）を $B$ 個まとめて 1 回で計算します。1 個ずつ計算するより、GPU の多数のコアを使いやすくなります。

In [ ]:
BATCH_SIZES = [1, 4, 16, 64]   # TODO：変えてみよう
MAT_N = 128
batch_rows = []
if HAS_TORCH:
    for d in DEVICES:
        sync = (lambda _r=None: torch.cuda.synchronize()) if d == "cuda" else None
        for bsz in BATCH_SIZES:
            g = torch.Generator().manual_seed(SEED)
            X = torch.randn(bsz, MAT_N, MAT_N, generator=g).to(d)
            Y = torch.randn(bsz, MAT_N, MAT_N, generator=g).to(d)
            r = measure(lambda: torch.bmm(X, Y), repeats=REPEATS, warmup=WARMUP, sync=sync)
            batch_rows.append({"device": LABEL[d], "batch": bsz, "n": MAT_N,
                               "compute_s": r["median"], "per_matrix_s": r["median"] / bsz})
    print(format_table(batch_rows, [("device", "device"), ("batch", "batch"), ("n", "n"),
                                    ("compute_s", "compute"), ("per_matrix_s", "per matrix")]))

## 6. 結果の可視化

**実際に測定できたデバイスだけ**を表示します。

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))

for (impl, dev) in sorted({(r["impl"], r["device"]) for r in size_rows}):
    pts = sorted((r["n"], r["compute_s"]) for r in size_rows if r["impl"] == impl and r["device"] == dev)
    xs, ys = zip(*pts)
    ls, mk = DEVICE_STYLE[dev]
    ax1.loglog(xs, np.array(ys) * 1e3, ls, marker=mk, ms=6, color=COLORS[impl], label=f"{impl} / {dev}")
ax1.set_xlabel("matrix size n (n x n)"); ax1.set_ylabel("compute time [ms]")
ax1.set_title("matmul: compute time vs size"); ax1.legend()

if batch_rows:
    for dev in sorted({r["device"] for r in batch_rows}):
        pts = [(r["batch"], r["per_matrix_s"]) for r in batch_rows if r["device"] == dev]
        xs, ys = zip(*pts)
        ls, mk = DEVICE_STYLE[dev]
        ax2.loglog(xs, np.array(ys) * 1e3, ls, marker=mk, ms=6, color=COLORS["PyTorch"], label=f"PyTorch / {dev}")
    ax2.set_xlabel("batch size"); ax2.set_ylabel("time per matrix [ms]")
    ax2.set_title(f"batched matmul ({MAT_N} x {MAT_N})"); ax2.legend()
else:
    ax2.set_axis_off(); ax2.text(0.5, 0.5, "PyTorch not available", ha="center")
plt.tight_layout(); plt.show()

In [ ]:
# 6-3. 転送時間と計算時間の内訳（PyTorch、測定できたデバイスのみ）
torch_rows = [r for r in size_rows if r["impl"] == "PyTorch"]
if torch_rows:
    labels = [f"{r['device']}\nn={r['n']}" for r in torch_rows]
    parts = [("to_device_s", "to device", "#8a8985"), ("compute_s", "compute", COLORS["PyTorch"]),
             ("to_host_s", "to host", "#2a78d6")]
    bottom = np.zeros(len(torch_rows))
    plt.figure(figsize=(12, 3.8))
    for key, name, color in parts:
        vals = np.array([r[key] for r in torch_rows]) * 1e3
        plt.bar(range(len(torch_rows)), vals, bottom=bottom, color=color, edgecolor="white", label=name)
        bottom += vals
    plt.xticks(range(len(torch_rows)), labels, fontsize=8)
    plt.yscale("log"); plt.ylabel("time [ms] (stacked, log)")
    plt.title("transfer vs compute (PyTorch)"); plt.legend()
    plt.show()

## 7. 結果の解釈（読み方のヒント）

- **小さい行列**：GPU の計算本体は速くても、転送やカーネル起動の固定コストのため、CPU の方が速いことがあります。
- **大きい行列**：$n^3$ の計算が支配的になり、GPU が有利になりやすい。
- **転送**：`to_device_s` と `to_host_s` は、毎回データを送る場合には計算時間に加わります。GPU の計算が速くても、転送込みでは CPU の方が速いことがあります。
- **CPU の NumPy と CPU の PyTorch**：同じ CPU でも、ライブラリ（内部で使う BLAS やスレッド数）が違うと時間が変わります。「デバイスの差」と「ライブラリの差」を区別してください。
- **バッチ**：小さな行列をまとめて計算すると、1 個あたりの時間が下がることがあります（特に GPU）。

## 8. 学生向け TODO

- [ ] **TODO 1**：行列サイズ `MATRIX_SIZES` を変更する（`MAX_MATRIX_N` まで）。
- [ ] **TODO 2**：バッチサイズ `BATCH_SIZES` を変更する。
- [ ] **TODO 3**：CPU と GPU の処理時間を、計算本体だけの場合と転送込みの場合で比べる。
- [ ] **TODO 4**：小規模計算で GPU が不利になる条件（行列サイズ）を探し、境目を求める。
- [ ] **TODO 5**：`DTYPE` を `np.float16`（GPU のみ）にして、時間と誤差を比べる。
- [ ] **TODO 6**：要素ごとの演算（`a * b + c`）でも CPU と GPU を比べ、行列積との違いを調べる。

## 9. 考察課題

1. GPU が CPU より速くなったのは、行列サイズがいくつ以上のときでしたか（GPU が使えた場合）。なぜ小さいサイズでは不利なのですか。
2. `torch.cuda.synchronize()` を使わずに測ると、時間はどう見えるはずですか。なぜ同期が必要なのですか。
3. 行列積（$O(n^3)$ の計算、$O(n^2)$ のデータ）と、要素ごとの演算（$O(n)$ の計算、$O(n)$ のデータ）では、GPU の効果はどちらが大きいと考えられますか。
4. GPU が使えなかった場合、CPU の NumPy と PyTorch の結果から何が言えますか。

## 10. 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 探索、第6章 ソート）
- 参考 Notebook：`ai-programming-2026-team-project-1-starter/A_cpu_gpu_tpu/A_cpu_gpu_tpu_benchmark/starter.ipynb`（MIT License, Copyright (c) 2026 Daigo Takeuchi）
- NumPy ドキュメント <https://numpy.org/doc/>
- PyTorch ドキュメント：CUDA semantics（非同期実行と同期）<https://pytorch.org/docs/stable/notes/cuda.html>
- JAX ドキュメント：Asynchronous dispatch、Just-in-time compilation <https://docs.jax.dev/>

- E2 リポジトリの `README.md`